# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [2]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [13]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)
df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [14]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [17]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [18]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [19]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [21]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.2190
Epoch 10/10 - Loss: 0.0765


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [22]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.770


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [23]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.77      0.74      0.76        96
    Positive       0.77      0.80      0.78       104

    accuracy                           0.77       200
   macro avg       0.77      0.77      0.77       200
weighted avg       0.77      0.77      0.77       200

Confusion Matrix:
[[71 25]
 [21 83]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [131]:
import numpy as np
import torch
from gensim.models import Word2Vec

X_train_tokens= [review.split() for review in X_train]
X_test_tokens= [review.split() for review in X_test]

w2v_model= Word2Vec(
    sentences= X_train_tokens,
    vector_size= 200,
    window= 5,
    min_count= 1,
    sg= 1
)

def review_to_vector(tokens, model, vector_size= 200):
    vectors= [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis= 0)

X_train_w2v= np.array([review_to_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v= np.array([review_to_vector(tokens, w2v_model) for tokens in X_test_tokens])

X_train_w2v_tensor= torch.FloatTensor(X_train_w2v)
X_test_w2v_tensor= torch.FloatTensor(X_test_w2v)
y_train_tensor= torch.FloatTensor(y_train.values).reshape(-1, 1)

print(f"Training Word2Vec tensor shape: {X_train_w2v_tensor.shape}")
print(f"Testing Word2Vec tensor shape: {X_test_w2v_tensor.shape}")

Training Word2Vec tensor shape: torch.Size([800, 200])
Testing Word2Vec tensor shape: torch.Size([200, 200])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [132]:
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

input_size= X_train_w2v_tensor.shape[1]

model_w2v= nn.Sequential(
    nn.Linear(input_size, 128), # Input Layer
    nn.ReLU(),
    
    nn.Linear(128, 64), # Hidden Layer 1
    nn.ReLU(),
    
    nn.Linear(64, 32), # Hidden Layer 2
    nn.ReLU(),
    
    nn.Linear(32, 1) # Output Layer
)

print(model_w2v)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [133]:
loss_function= nn.BCEWithLogitsLoss()
optimizer= torch.optim.Adam(model_w2v.parameters(), lr= 0.001)

epochs= 15

for epoch in range(epochs):
    model_w2v.train()
    
    # Clear gradients from the previous step
    optimizer.zero_grad()
    
    # The forward pass
    outputs= model_w2v(X_train_w2v_tensor)
    
    # Calculate loss
    loss= loss_function(outputs, y_train_tensor)
    
    # The backpropagation pass
    loss.backward()
    
    # Weights update
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/15 - Loss: 0.6959
Epoch 10/15 - Loss: 0.6943
Epoch 15/15 - Loss: 0.6934


In [134]:
model_w2v.eval()

with torch.no_grad():
    test_logits= model_w2v(X_test_w2v_tensor)
    test_probabilities= torch.sigmoid(test_logits)
    predictions_w2v= (test_probabilities >= 0.5).int().numpy().flatten()

accuracy_w2v= accuracy_score(y_test, predictions_w2v)
print(f"\nTest accuracy: {accuracy_w2v:.2f}")

print("\nClassification report")
print(classification_report(y_test, predictions_w2v, target_names= ["Negative", "Positive"]))

print("\nConfusion matrix")
print(confusion_matrix(y_test, predictions_w2v))


Test accuracy: 0.51

Classification report
              precision    recall  f1-score   support

    Negative       0.49      0.93      0.64        96
    Positive       0.65      0.12      0.21       104

    accuracy                           0.51       200
   macro avg       0.57      0.53      0.43       200
weighted avg       0.58      0.51      0.42       200


Confusion matrix
[[89  7]
 [91 13]]


# Lab 6: Deep Learning for NLP

## Student Information

**Name:** Muhannad Almutlaq  
**ID:** 2240006060

### Experimentation & Key Takeaways

* **Binary Sentiment Classification:** Built deep learning models to classify 1,000 Yelp restaurant reviews into Negative (`0`) and Positive (`1`) sentiment categories.
* **Text Preprocessing & Data Splitting:** Cleaned raw review text using Regular Expressions (`re`) to convert text to lowercase, remove punctuation and special characters, and strip extra whitespace, followed by splitting the dataset into 80% training and 20% testing sets.
* **TF-IDF vs. Word2Vec Representation:** Converted unstructured reviews into numerical features first using `TfidfVectorizer` (`max_features= 1000`) and subsequently by training a Skip-Gram (`sg= 1`) Word2Vec model (`vector_size= 200`, `window= 5`, `min_count= 1`) to represent each review as a single vector.
* **PyTorch Tensor Conversion:** Transformed the extracted numerical feature vectors and sentiment labels into PyTorch `FloatTensor` objects for neural network compatibility.
* **Feedforward Neural Network Architectures:** Designed multi-layer neural networks using `torch.nn.Sequential`, `nn.Linear`, and `nn.ReLU` activation functions, experimenting with both a two-hidden-layer baseline (`64 → 32 → 1`) and a deeper three-hidden-layer architecture (`128 → 64 → 32 → 1`).
* **Loss Calculation & Backpropagation:** Trained the neural networks across multiple epochs using `BCEWithLogitsLoss` for numerically stable binary cross-entropy computation and the `Adam` optimizer to update model weights via backpropagation.
* **Model Evaluation & Metrics:** Applied `torch.sigmoid` to map raw output logits to probabilities with a `0.5` decision threshold, assessing model performance using Test Accuracy, Confusion Matrices, and Classification Reports (Precision, Recall, and F1-score).
